# 2.2 Обучение POS-регрессии


In [ ]:
# PROJECT_ROOT — ветка current, в том числе при запуске Jupyter из общего каталога.
from pathlib import Path
import sys, os
_branch = 'current'
_cwd = Path.cwd().resolve()
_candidates = [_cwd, _cwd / _branch, _cwd.parent / _branch]
PROJECT_ROOT = next((x for x in _candidates if x.name == _branch and (x / 'srcc' / 'nbc_project').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f'Не найдена папка {_branch}/srcc/nbc_project; запустите из ветки или её родителя')
SRC_PATH = PROJECT_ROOT / 'srcc'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))
os.chdir(PROJECT_ROOT)
print('PROJECT_ROOT:', PROJECT_ROOT)
EXPERIMENT_NAME = '202610_current'
RUN_TAG = '202610_current'


In [ ]:
import json
from pathlib import Path
import pandas as pd

from nbc_project.training_workflow import (
    TrainJobConfig,
    run_jobs,
    summarize_results,
    run_optional_rolling,
    run_optional_visuals,
)

with open(PROJECT_ROOT / 'configs' / 'training_jobs.json', encoding='utf-8') as f:
    TRAINING_CONFIG = json.load(f)

DEFAULT_SPLIT = TRAINING_CONFIG['default_split']
DEFAULT_SPLIT

## Конфигурация обучения

In [ ]:
configs = [
    TrainJobConfig(**{**DEFAULT_SPLIT, **cfg})
    for cfg in TRAINING_CONFIG['pos_regression']
]

for cfg in configs:
    print(cfg)
# Split/output конкретного эксперимента
for cfg in configs:
    cfg.train_end_exclusive='2026-10-01'
    cfg.valid_month='2026-10-01'
    cfg.test_month=None
    cfg.split_suffix='202610_current'
    cfg.legacy_test_suffix=None
    cfg.output_split_dir=str(PROJECT_ROOT / 'experiment_outputs' / '202610_current' / 'splits' / 'pos')
    cfg.model_output_dir=str(PROJECT_ROOT / 'experiment_outputs' / '202610_current' / 'models' / 'pos')
configs

for cfg in configs:
    segment = 'T' if cfg.model_key.endswith('_T') else 'C'
    cfg.model_filename = f"model_{segment}_202610_current.cbm"
configs

# current берёт фиксированные CSV только из общей подготовки common/1.data_fm.
COMMON_ROOT = PROJECT_ROOT.parent / 'common'
for cfg in configs:
    cfg.input_path = str(COMMON_ROOT / cfg.input_path)
    if not Path(cfg.input_path).is_file():
        raise FileNotFoundError(f'Нет входного CSV: {cfg.input_path}; выполните common/1.data_fm')


### Опционально: dynamic feature selection — v6



In [ ]:
USE_DYNAMIC_FS = False
# Победители benchmark задаются отдельно для T/C.
FS_BY_MODEL = {
    'pos_T': 'psi_metric_rfe_1se',
    'pos_C': 'psi_metric_rfe_1se'
}
DYNAMIC_FS_PRESETS = {
    'psi_importance_auto_n': {'method':'psi_importance','psi_threshold':0.20,'psi_reference_months':3,'auto_top_n':True,'top_n_candidates':[100,150,200,250,300],'sample_rows':150_000,'selector_iterations':200,'report_dir':'reports/feature_selection'},
    'psi_metric_rfe_1se': {'method':'psi_metric_rfe','psi_threshold':0.20,'psi_reference_months':3,'rfe_algorithm':'loss_function_change','rfe_steps':5,'metric_rfe_rule':'one_se','metric_rfe_min_features':30,'metric_rfe_drop_fraction':0.15,'metric_rfe_min_drop':10,'metric_rfe_max_models':16,'metric_rfe_bootstrap_repeats':60,'metric_rfe_one_se_multiplier':1.0,'metric_rfe_eval_rows':50_000,'sample_rows':150_000,'selector_iterations':180,'report_dir':'reports/feature_selection'},
    'psi_permutation_ci': {'method':'psi_permutation_ci','psi_threshold':0.20,'psi_reference_months':3,'sample_rows':150_000,'selector_iterations':180,'permutation_repeats':2,'permutation_eval_rows':30_000,'permutation_ci_max_features':None,'permutation_ci_bootstrap_repeats':30,'permutation_ci_bootstrap_rows':3_000,'permutation_ci_confidence':0.95,'permutation_ci_min_importance':0.0,'report_dir':'reports/feature_selection'},
    'psi_null_importance': {'method':'psi_null_importance','psi_threshold':0.20,'psi_reference_months':3,'sample_rows':150_000,'selector_iterations':150,'null_importance_repeats':10,'null_importance_quantile':0.95,'null_importance_max_features':None,'report_dir':'reports/feature_selection'},
}
CORRELATION_PREFILTER={'drop_correlated_features':True,'correlation_threshold':0.95,'correlation_method':'pearson','correlation_max_rows':20_000,'correlation_min_unique_values':20}
for _cfg in DYNAMIC_FS_PRESETS.values(): _cfg.update(CORRELATION_PREFILTER)

FULL_DATA_BY_SEGMENT={'T':'data/datasets/mae_nbc_fm_loyalty_dataset_modular_full','C':'data/datasets/mae_nbc_fm_noloyalty_dataset_modular_full'}
if USE_DYNAMIC_FS:
    for cfg in configs:
        segment='T' if cfg.model_key.endswith('_T') else 'C'
        preset=FS_BY_MODEL[cfg.model_key]
        cfg.input_path=FULL_DATA_BY_SEGMENT[segment]
        cfg.ignore_cols=list(dict.fromkeys(cfg.ignore_cols+['period_dt','n_treat','rn']))
        cfg.feature_selection=DYNAMIC_FS_PRESETS[preset].copy()
        cfg.model_filename=cfg.model_filename.replace('.cbm', f'_{preset}.cbm')
else:
    for cfg in configs:
        cfg.feature_selection=None
print('USE_DYNAMIC_FS=',USE_DYNAMIC_FS,'FS_BY_MODEL=',FS_BY_MODEL)
configs


## Обучение T/C-сегментов

In [ ]:
results = run_jobs(configs)

## Сводка метрик

In [ ]:
summary = summarize_results(results)
summary

## 4. Визуализации по таргетам, моделям, метрикам и feature importance

Блок строит презентационные графики для каждой модели/сегмента: target distribution, boxplot таргета, metrics bar, диагностику prediction/fact или ROC/PR/calibration, confusion matrix для binary и feature importance.

In [ ]:
RUN_VISUALS = True

if RUN_VISUALS:
    visuals = run_optional_visuals(
        results,
        output_dir='reports/training_visuals',
    )
    visuals


## 5. Rolling train / walk-forward backtest

Схема проверки сохранена:

- train: все месяцы до `train_end` включительно;
- valid: следующий месяц;
- test: через месяц после valid, то есть `valid + 2 календарных месяца`;
- OOT heatmap: оценка на будущих месяцах.

По умолчанию используется облегчённый CatBoost для rolling, чтобы блок можно было запускать в ноутбуке. Для финального прогона увеличь `iterations`.

In [ ]:
RUN_ROLLING_BACKTEST = True

ROLLING_MODEL_PARAMS = {
    'iterations': 300,
    'learning_rate': 0.05,
    'depth': 6,
    'l2_leaf_reg': 5,
    'random_seed': 42,
    'early_stopping_rounds': 50,
    'verbose': 100,
}

if RUN_ROLLING_BACKTEST:
    rolling = run_optional_rolling(
        results,
        output_dir='reports/rolling_backtest',
        min_train_months=3,
        model_params=ROLLING_MODEL_PARAMS,
    )
    rolling


## 6. Итоговые артефакты

После Run All будут сохранены:

- модели и metadata в директории из `model_output_dir`;
- split-файлы train/valid/test;
- `*_feature_importance.csv`;
- графики в `reports/training_visuals/<model_key>/`;
- rolling metrics/heatmap в `reports/rolling_backtest/<model_key>/`.

In [ ]:
{
    'summary': summary,
    'visuals_dir': 'reports/training_visuals',
    'rolling_dir': 'reports/rolling_backtest' if RUN_ROLLING_BACKTEST else None,
}
